# Module 2: Deploy + Context Hub

> Part of the **Modular Workshops** series. Standalone, ~25 min.

Two steps to take the agent to production:

- **Context Hub** — push the agent's context (`AGENTS.md` + skills) to LangSmith as **versioned, shareable repos**, so it's centralized and reusable instead of baked into one codebase.
- **Deployment** — ship the agent to **LangSmith Deployments** with the `langgraph` CLI: 30+ endpoints, persistence, HITL, and Studio out of the box.

Context first, then deploy.

## Setup


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:        ", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:           ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:           ", bool(os.environ.get("TAVILY_API_KEY")))

LANGSMITH_API_KEY set:         True
OPENAI_API_KEY set:            True
TAVILY_API_KEY set:            True


---
# Part 1. Context Hub — Centralize the Agent's Context

An agent's *context* — its `AGENTS.md` instructions and its skills — matters as much as its code. **LangSmith Context Hub** stores that context as **versioned, shareable repos**: push it once, and any deployment (or teammate, or *other* agent) pulls the same source of truth.

We push it in two pieces — each **skill** as its own reusable repo, then the **agent repo** that links them — and read it back. At deploy time the agent loads this context via deepagents' `ContextHubBackend` (mounted at `/context/`).

### 1.1 Push each skill as a reusable repo

`push_skill` creates (or versions) a standalone skill repo. Because it's independent, **any** agent can link it — no copy-paste.

In [2]:
from pathlib import Path
from langsmith import Client
from langsmith.schemas import FileEntry, SkillEntry

client = Client()
agent_dir = project_root / "agents" / "deep_agent"

skill_urls = {}
for skill_dir in sorted((agent_dir / "skills").iterdir()):
    skill_md = skill_dir / "SKILL.md"
    if not skill_md.exists():
        continue
    skill_urls[skill_dir.name] = client.push_skill(
        skill_dir.name,
        files={"SKILL.md": FileEntry(content=skill_md.read_text())},
        description=f"Reusable '{skill_dir.name}' skill.",
        tags=["workshop"],
    )

for name, url in skill_urls.items():
    print(f"skill '{name}': {url}")

skill 'linkedin-post': https://smith.langchain.com/context/linkedin-post/ea881d84?organizationId=a3866f07-2cf5-4e9c-a287-59ed817c2ecd
skill 'twitter-post': https://smith.langchain.com/context/twitter-post/ea881d84?organizationId=a3866f07-2cf5-4e9c-a287-59ed817c2ecd


### 1.2 Push the agent context (linking the skills)

`push_agent` uploads `AGENTS.md` and **links** the skill repos with `SkillEntry` instead of inlining them — so a skill update propagates to every agent that links it.

In [3]:
agent_url = client.push_agent(
    "modular-workshops-research-agent",
    files={
        "AGENTS.md": FileEntry(content=(agent_dir / "AGENTS.md").read_text()),
        **{f"skills/{name}": SkillEntry(repo_handle=name) for name in skill_urls},
    },
    description="Research assistant used in the Modular Workshops deploy module.",
    tags=["workshop", "research"],
)
print("agent context:", agent_url)

agent context: https://smith.langchain.com/context/modular-workshops-research-agent/805a628a?organizationId=a3866f07-2cf5-4e9c-a287-59ed817c2ecd


### 1.3 Pull it back — and how the deployed agent uses it

`pull_agent` fetches the latest commit (pin a `version=` or tag for reproducibility). At deploy time, `agents/deep_agent/agent.py` mounts a Context Hub repo at `/context/` through deepagents' `ContextHubBackend`, so the running agent reads this same context — centralized and versioned, not baked into the image.

In [4]:
pulled = client.pull_agent("modular-workshops-research-agent")
print("commit:", pulled.commit_hash)
print("files:", list(pulled.files))

commit: 805a628a3b25e87888e0908d7d202895e701aaef64f7e97d6e390bd6b5f999ca
files: ['AGENTS.md', 'skills/linkedin-post', 'skills/linkedin-post/SKILL.md', 'skills/twitter-post', 'skills/twitter-post/SKILL.md']


---
# Part 2. Deploy — Ship the Agent

With the context in the Hub, ship the deployment itself. We deploy the agent in `agents/deep_agent/` (and your own `my_agent/`) to **LangSmith Deployments** with the `langgraph` CLI. The deployed agent can pull its context from the Hub at runtime via `ContextHubBackend`.

## 2.1 Project structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. We already have one — `langgraph.json` at the workshop root registers the deep agent at `agents/deep_agent/agent.py`. You'll add a second graph — **your own agent** — in §2.3, and a single deploy ships both.


In [5]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("langgraph.json (workshop root)")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


langgraph.json (workshop root)
---
deep_agent/
  deepagents.toml
  agent.py
  AGENTS.md
  skills/
    linkedin-post/
      SKILL.md
    twitter-post/
      SKILL.md


In [6]:
# langgraph.json -- the deploy configuration
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


langgraph.json:
{
  "dependencies": [
    "."
  ],
  "graphs": {
    "deep_agent": "./agents/deep_agent/agent.py:agent",
    "my_agent": "./my_agent/agent.py:agent"
  },
  "env": ".env"
}

AGENTS.md:
# Research Assistant

You are an expert research assistant that can search the web, synthesize findings, and produce polished reports and content.

## Workflow

1. **Plan** — Use `write_todos` to break the task into steps
2. **Research** — Delegate research to the `research-agent` using the `task()` tool
3. **Synthesize** — Combine findings into a comprehensive report
4. **Write** — Save the final report to `/final_report.md`
5. **Remember** — Save key takeaways to `/memories/research_notes.md` for future reference

## Rules

- Delegate research to the research-agent rather than searching directly
- After receiving research results, synthesize and write the report yourself
- Consolidate citations — each unique URL gets one number [1], [2], [3]
- End reports with a Sources section listing a

The agent itself lives in `agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"deep_agent": "./agents/deep_agent/agent.py:agent"`.


## 2.2 Local development

Three CLI commands you'll use (all run from the workshop root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

# Deploy to LangSmith (beta)
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to your local server.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 2.3 Create your own deep agent

Your turn. `my_agent/agent.py` is a blank deep-agent scaffold — a `create_deep_agent` with one search tool and a research subagent, plus commented stubs for extra tools, subagents, backends, middleware, skills, and human-in-the-loop. Make it yours:

1. Open **`my_agent/agent.py`** and edit the system prompt, add tools, wire up subagents — whatever your use case needs. (`my_agent/AGENTS.md` is its identity file.)
2. It's **already registered** in `langgraph.json` as the `my_agent` graph, so you can visualize it with the tools from the last two sections:
   - `langgraph dev --port 2024` (§2.2) picks it up — open **Studio** and pick `my_agent` to step through it.
   - the deploy step (§2.5) ships it **alongside** `deep_agent` in one deployment.

Run the cell to smoke-test that your agent builds and responds:


In [7]:
# Reload so edits to my_agent/agent.py are picked up without restarting the kernel.
import importlib, my_agent.agent
importlib.reload(my_agent.agent)
from my_agent.agent import agent as my_agent_graph

result = my_agent_graph.invoke(
    {"messages": [{"role": "user", "content": "In one sentence, introduce yourself and what you can do."}]}
)
print(result["messages"][-1].content)


I’m an AI assistant that can answer questions, write and edit code or files, search and summarize information, and help plan or complete tasks.


## 2.4 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch config and import errors before deploying.


In [8]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


Configuration file /Users/christineastoria/demos/deep-agents-research-workshop/langgraph.json is valid. (2 graphs found)


## 2.5 Deploy to LangSmith (optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` ships **every graph** in `langgraph.json`, so `deep_agent` and your `my_agent` go out together in one deployment. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.


> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
!cd "{project_root}" && langgraph deploy --name modular-workshops-deep-agent --no-input


Note: 'langgraph deploy' is in beta. Expect frequent updates and improvements.

⚠️  Security Recommendation: Consider switching to Wolfi Linux for enhanced security.
   Wolfi is a security-oriented, minimal Linux distribution designed for containers.
   To switch, add '"image_distro": "wolfi"' to your langgraph.json config file.
Skipping reserved env var: LANGSMITH_ENDPOINT
Skipping reserved env var: LANGSMITH_API_KEY
Skipping reserved env var: LANGSMITH_PROJECT
1. Looking up deployment 'modular-workshops-deep-agent'
   Found existing deployment (ID: c2a2b39f-c899-4040-97f4-7bbba41061ff)
2. Building image
/ Building... (1m 47s)

## 2.6 What you get with LangSmith Deployments

Once deployed, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |


---
## Optional · Managed Deep Agents (beta)

> **Beta, US-only** — needs beta access (see the [overview](https://docs.langchain.com/langsmith/python/managed-deep-agents-overview) to request it). Skip this section if you don't have it. The cells below actually scaffold and deploy an MDA project.

**How is this different from the `langgraph deploy` you just ran?** Both ship to LangSmith and expose the same hosted endpoints. The difference is **how much of the agent *you* assemble vs. how much the platform manages.**

In Part 2 you **built** the agent: you wrote `create_deep_agent(...)`, wired up the backend, store, memory, skills, subagents, and HITL yourself, and registered it in `langgraph.json`. `langgraph deploy` then packaged **your** code into a Docker image and hosted it — maximum control, your graphs, your server config.

Managed Deep Agents flips that: you never call `create_deep_agent` or touch `langgraph.json`. You **declare** the agent — `define_deep_agent(name, model, tools)` plus a folder of conventions (`instructions.md`, `skills/`, `tools/`, `sandbox/`, …) — and LangSmith **builds, wires, and operates the entire deep-agent harness** (backend, store, checkpointer, memory, sandbox) for you. No Docker, no server config; `instructions.md` and skills are Context Hub-backed, so you edit them without redeploying.

| | **LangGraph CLI** (Part 2) | **Managed Deep Agents** |
|---|---|---|
| You write | `create_deep_agent(...)` — you assemble the harness | `define_deep_agent(...)` + a config folder |
| Deploy config | your own `langgraph.json` (any # of graphs) | none — the folder *is* the config |
| Harness (store, memory, sandbox) | you own and wire it | the platform provides + runs it |
| Ship it | `langgraph deploy` (builds a Docker image) | `mda deploy` (no Docker) |
| Reach for it when | full control / custom graphs (`deep_agent` + `my_agent`) | the fastest hosted deep agent |

**In one line:** *LangGraph CLI — you build the agent, the platform hosts it. MDA — you describe the agent, the platform builds and runs it.*

In [5]:
# Install the Managed Deep Agents CLI (`mda`). One-time; safe to re-run.
# If you later see "mda: command not found", the uv tool bin isn't on PATH:
# run `uv tool update-shell` in a terminal and restart the kernel.
!uv tool install managed-deepagents
!mda --version

`managed-deepagents` is already installed
mda 0.5.3 (8eb4c07 2026-08-18)


**Step 1 · Scaffold the project.** `mda init` creates a self-contained MDA project (its own `agent.py`, `instructions.md`, `.env`, `sandbox/`, …). We scaffold it in a **sibling folder** so it stays out of the workshop repo.

In [6]:
MDA_PARENT = project_root.parent            # scaffold beside the workshop repo, not inside it
MDA_DIR = MDA_PARENT / "research-assistant"

if MDA_DIR.exists():
    print(f"{MDA_DIR} already exists — skipping init (delete it to re-scaffold)")
else:
    !cd "{MDA_PARENT}" && mda init research-assistant

print("MDA project:", MDA_DIR)

/Users/christineastoria/demos/research-assistant already exists — skipping init (delete it to re-scaffold)
MDA project: /Users/christineastoria/demos/research-assistant


**Step 2 · Configure it.** Set the model + a built-in web-search tool in `agent.py`, write the system prompt to `instructions.md`, and drop your keys into the project's own `.env` — pulled from the already-loaded environment, never hardcoded.

> MDA runs the model on **managed infrastructure** using the keys in *this* `.env`. Use a real provider key here — a local gateway/proxy key won't be reachable from LangSmith's servers.

In [7]:
import os, textwrap

# agent.py — OpenAI's built-in, server-side web search (no extra key or install).
(MDA_DIR / "agent.py").write_text(textwrap.dedent('''\
    from managed_deepagents import define_deep_agent

    agent = define_deep_agent(
        name="research-assistant",
        model="openai:gpt-5.5",
        tools=[{"type": "web_search"}],
    )
'''))

# instructions.md — the system prompt (synced to Context Hub on deploy).
(MDA_DIR / "instructions.md").write_text(textwrap.dedent('''\
    # Research assistant

    You are a careful research assistant. Use web search to find sources,
    take notes, and return concise answers with citations.
'''))

# .env — keys come from the environment, not hardcoded into the notebook.
missing = [k for k in ("LANGSMITH_API_KEY", "OPENAI_API_KEY") if not os.environ.get(k)]
if missing:
    print("WARNING missing env vars:", missing, "- set them before deploying")
with open(MDA_DIR / ".env", "w") as f:
    for k in ("LANGSMITH_API_KEY", "OPENAI_API_KEY"):
        f.write(f"{k}={os.environ.get(k, '')}\n")

print("Wrote agent.py, instructions.md, .env ->", MDA_DIR)

Wrote agent.py, instructions.md, .env -> /Users/christineastoria/demos/research-assistant


**Step 3 · Try it locally (optional).** `mda dev` is a **blocking** dev server (like `langgraph dev`) — run it in a terminal, not a notebook cell:

```bash
cd ../research-assistant
mda dev .
```

It loads `.env`, starts a local Agent Server, and opens the agent in LangSmith Studio. Send a prompt like *"What were the main announcements from the latest LangChain release?"* and watch the web-search tool call in the trace.

**Step 4 · Deploy.** `mda deploy` packages the project, runs it on **LangSmith-managed infrastructure**, syncs `instructions.md` to Context Hub, and prints a **dashboard URL**. Provisioning takes a few minutes; open the URL and send the same question to confirm the hosted agent answers with a search call.

In [ ]:
# Actually deploy to LangSmith managed infra. Prints a dashboard URL when ready.
!cd "{MDA_DIR}" && mda deploy .

┌   mda deploy 
│
●  Project: /Users/christineastoria/demos/research-assistant
│  
◐  BUILDING…                                                                    │                                                                               ◒  Triggering remote build                                                      

## Recap

| Pillar | What | How |
|---|---|---|
| **Context Hub** | Reusable skill repos | `client.push_skill(name, files={...})` |
| **Context Hub** | Versioned agent context that links skills | `client.push_agent(..., files={"skills/x": SkillEntry(...)})` |
| **Deploy** | Deployable graph config | `langgraph.json` (registers `deep_agent` + `my_agent`) |
| **Deploy** | Ship it | `langgraph deploy --name <your-deployment-name>` |
| **Deploy** *(optional)* | Fully-managed alternative | `mda deploy` (Managed Deep Agents, beta) |

**The production pattern:** push context to the Hub once, register your graphs in `langgraph.json`, then one `langgraph deploy` puts them behind a managed server with 30+ endpoints — reading their context from the Hub. Or skip the infra entirely with **Managed Deep Agents**.

**Next:** Module 3 — LangSmith (tracing, querying traces, offline + online evals, annotation queues).
